# Kerendia Paid TRX Overlap Analysis

**Purpose:** Compute M4 (Kerendia Paid TRX > 0 from raw IQVIA FCT_RX) and cross it with the 59,635 improved HCPs from the NPI/BH_ID migration overlap analysis.

**M4 Definition:** Distinct PROVIDER_IDs from `PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX` joined with `STG_CAR_IQV_LAAD_DIM_PRD` where `CLAIM_TYPE='PD'` and `PRODUCT_GROUP='KERENDIA'`, 24-month lookback (2024-10-01 to 2026-09-30).

**Source:** Cell 30 of `05_HCP_Universe_V5_NBRx_Analysis`

In [0]:
%run "/Users/wenjie.chen@bayer.com/kerendia_data_needs/LAAD based LRx Pipeline/Ops Pipeline/00_connection"

In [0]:
from pyspark.sql import functions as F

CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
PACE = f"`{CATALOG}`.pace_ker_in_house_schema"

LOOKBACK_START = "2024-10-01"
LOOKBACK_END   = "2026-09-30"

print(f"Lookback: {LOOKBACK_START} to {LOOKBACK_END}")

In [0]:
# ============================================================
# M4: Kerendia writers from raw IQVIA LAAD FCT_RX (Snowflake)
# Replicates NBA 3.0 logic: FCT_RX JOIN DIM_PRD on NDC_CD
# CLAIM_TYPE='PD', PRODUCT_GROUP='KERENDIA', 24-mo lookback
# Source: Cell 30 of 05_HCP_Universe_V5_NBRx_Analysis
# ============================================================
query_m4 = f'''
    SELECT DISTINCT A.PROVIDER_ID
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    USING (NDC_CD)
    WHERE A.CLAIM_TYPE = 'PD'
    AND A.SVC_DT >= '{LOOKBACK_START}'
    AND A.SVC_DT <= '{LOOKBACK_END}'
    AND UPPER(TRIM(B.PRODUCT_GROUP)) = 'KERENDIA'
'''

print("Querying Snowflake for M4: Kerendia Paid TRX...")
m4_raw = Get_Data_Snowflakes(query_m4)
m4_writers = m4_raw.select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID")).distinct()
m4_ct = m4_writers.count()
print(f"M4: Kerendia Paid TRX providers (24-mo): {m4_ct:,}")

# Save to temp view for downstream analysis
m4_writers.createOrReplaceTempView("m4_kerendia_trx")

In [0]:
# ============================================================
# Reproduce the 59,635 improved HCPs from Step 5 overlap
# Old V6: MIN(BH_ID_VIA_NPI) per PROVIDER_ID
# New UC: 2-table CASE cascade
# Improved = Old was PROVIDER_ID default, New is BH*
# ============================================================
V6_TABLE = f"`{CATALOG}`.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v6"

# Old V6 dedup (MIN picks PROVIDER_ID over BH* since '0'-'9' < 'B')
old_v6 = spark.sql(f"""
    SELECT PROVIDER_ID, MIN(BH_ID_VIA_NPI) AS OLD_BH_ID
    FROM {V6_TABLE}
    GROUP BY PROVIDER_ID
""")
old_v6.createOrReplaceTempView("old_v6")

# New UC cascade
new_uc = spark.sql(f"""
    WITH base AS (
        SELECT DISTINCT PROVIDER_ID FROM {PACE}.lrx_nbrx WHERE NBRX > 0
    ),
    demo AS (
        SELECT PROVIDER_ID, FIRST(BH_ID) AS BH_ID, FIRST(HCP_NPI_ID) AS NPI
        FROM {PACE}.pha_hcp_demographics_np_pa_aligned_mar2026
        GROUP BY PROVIDER_ID
    ),
    fallback AS (
        SELECT NPI_NUM, MIN(BH_ID) AS BH_ID
        FROM {PACE}.mdm_stg_hcp_prfl
        WHERE NPI_NUM IS NOT NULL AND TRIM(NPI_NUM) != ''
          AND BH_ID IS NOT NULL AND TRIM(BH_ID) != ''
        GROUP BY NPI_NUM
    )
    SELECT b.PROVIDER_ID,
           CASE
               WHEN d.BH_ID IS NOT NULL AND TRIM(d.BH_ID) != '' THEN d.BH_ID
               WHEN d.NPI IS NOT NULL AND TRIM(d.NPI) != '' AND f.BH_ID IS NOT NULL THEN f.BH_ID
               ELSE b.PROVIDER_ID
           END AS NEW_BH_ID
    FROM base b
    LEFT JOIN demo d ON b.PROVIDER_ID = d.PROVIDER_ID
    LEFT JOIN fallback f ON d.NPI = f.NPI_NUM
""")
new_uc.createOrReplaceTempView("new_uc")

# Identify improved HCPs
improved = spark.sql("""
    SELECT o.PROVIDER_ID, o.OLD_BH_ID, n.NEW_BH_ID
    FROM old_v6 o
    INNER JOIN new_uc n ON o.PROVIDER_ID = n.PROVIDER_ID
    WHERE (o.OLD_BH_ID NOT LIKE 'BH%' OR o.OLD_BH_ID IS NULL)
      AND n.NEW_BH_ID LIKE 'BH%'
""")
improved.createOrReplaceTempView("improved_hcps")
print(f"Improved HCPs (Old=PROVIDER_ID, New=BH*): {improved.count():,}")

In [0]:
# NBA 3.0 Modeling Universe
nba3_universe = spark.read.format("delta").load(
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
    "hcp_level_model/dev_folder_3_1/notebook_output/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"
)
nba3_universe.createOrReplaceTempView("nba3_universe")
print(f"NBA 3.0 Universe: {nba3_universe.count():,} HCPs")

# M2: Kerendia NBRX > 0 from V6 (last 2 years)
m2_writers = spark.sql(f"""
    SELECT DISTINCT PROVIDER_ID
    FROM {V6_TABLE}
    WHERE PRODUCT_GROUP = 'KERENDIA' AND NBRX >= 1
      AND CAST(WEEK_ENDING_DATE AS INT) >= 20241001
""")
m2_writers.createOrReplaceTempView("m2_kerendia_nbrx")
print(f"M2: Kerendia NBRX>0 (V6, 24-mo): {m2_writers.count():,}")

In [0]:
# ============================================================
# Main analysis: cross 59,635 improved HCPs with:
# - NBA 3.0 Universe (in/out)
# - M2: Kerendia NBRX > 0 (V6 based)
# - M4: Kerendia Paid TRX > 0 (raw FCT_RX Snowflake)
# ============================================================
result = spark.sql("""
    SELECT
        CASE WHEN u.PROVIDER_ID IS NOT NULL THEN 'In NBA 3.0 Modeling Universe'
             ELSE 'Not in NBA 3.0 Universe' END AS category,
        COUNT(DISTINCT i.PROVIDER_ID) AS total_count,
        COUNT(DISTINCT CASE WHEN m2.PROVIDER_ID IS NOT NULL THEN i.PROVIDER_ID END) AS kerendia_nbrx_gt0,
        COUNT(DISTINCT CASE WHEN m4.PROVIDER_ID IS NOT NULL THEN i.PROVIDER_ID END) AS kerendia_paid_trx_gt0
    FROM improved_hcps i
    LEFT JOIN nba3_universe u ON i.PROVIDER_ID = u.PROVIDER_ID
    LEFT JOIN m2_kerendia_nbrx m2 ON i.PROVIDER_ID = m2.PROVIDER_ID
    LEFT JOIN m4_kerendia_trx m4 ON i.PROVIDER_ID = m4.PROVIDER_ID
    GROUP BY CASE WHEN u.PROVIDER_ID IS NOT NULL THEN 'In NBA 3.0 Modeling Universe'
                  ELSE 'Not in NBA 3.0 Universe' END
    ORDER BY total_count DESC
""")

display(result)

In [0]:
# ============================================================
# Summary with totals row
# ============================================================
from pyspark.sql.functions import sum as _sum, lit

summary = result.unionByName(
    result.agg(
        lit("TOTAL").alias("category"),
        _sum("total_count").alias("total_count"),
        _sum("kerendia_nbrx_gt0").alias("kerendia_nbrx_gt0"),
        _sum("kerendia_paid_trx_gt0").alias("kerendia_paid_trx_gt0")
    )
)

display(summary)

# Print for easy copy
for row in summary.collect():
    pct = f"({100*row['total_count']/59635:.1f}%)" if row['category'] != 'TOTAL' else ''
    print(f"{row['category']:35s}  Count: {row['total_count']:>8,} {pct:>8s}  "
          f"NBRX>0: {row['kerendia_nbrx_gt0']:>6,}  Paid TRX>0: {row['kerendia_paid_trx_gt0']:>6,}")